# contiguous-layout — worked example 1: Derive the contiguous strides of a 4-D tensor from its shape

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `contiguous-layout`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

A **contiguous (row-major)** tensor stores its elements so that the *last* axis varies fastest in memory. The stride of an axis is the number of elements you skip to advance one step along it: `stride[-1] = 1`, and each earlier axis's stride is the product of all sizes to its right. So strides are just the reversed running product of the shape.

## Worked solution

We want the strides PyTorch would assign to a fresh contiguous tensor of shape `(2, 5, 3, 4)`, computed purely from the shape.

**Step 1 — start from the last axis.** The innermost axis always has stride `1`: consecutive elements along it are literally adjacent in memory. So we begin with a running product `running = 1`.

**Step 2 — walk the shape from right to left.** For each axis (taken in reverse), the current `running` value *is* that axis's stride, because to step once along axis `k` you must skip over an entire block of all the axes to its right. After recording the stride, multiply `running` by that axis's size to grow the block for the next (more-outer) axis.

**Step 3 — reverse back.** We collected strides inner-to-outer, so we flip them to match the original axis order.

For `(2, 5, 3, 4)`: stride for axis 3 is `1`; then `running = 4`, so axis 2 is `4`; then `running = 12`, axis 1 is `12`; then `running = 60`, axis 0 is `60`. Result `(60, 12, 4, 1)`. This matches `t.zeros((2,5,3,4)).stride()` exactly — the reversed-prefix-product *is* the contiguous layout rule.

In [ ]:
def contiguous_strides(shape: tuple) -> tuple:
    strides = []
    running = 1
    for dim in reversed(shape):
        strides.append(running)
        running *= dim
    return tuple(reversed(strides))

shape = (2, 5, 3, 4)
predicted = contiguous_strides(shape)
actual = tuple(t.zeros(shape).stride())
print("predicted:", predicted)
print("actual:   ", actual)
print("match:", predicted == actual)